In [1]:
import os
import json
import torch
import gc
import time
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv, find_dotenv

_env_file = find_dotenv()
REPO_DIR = Path(_env_file).parent.resolve()
load_dotenv(_env_file)
ROOT = (REPO_DIR / os.getenv("ROOT")).resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.experiment_notebook import setup_experiment, fine_tune_and_evaluate
from src.utils.utils import set_all_seeds
from src.fairness_losses import get_fairness_loss

In [2]:
experiment = "20260426_025714"  # "" = nou experiment | "20260414_153000" = reprendre un existent

In [3]:
# ── Configuració del sweep ──────────────────────────────────────────────────
N_RUNS          = 5                          # models per configuració
BASE_SEED       = 42                         # seeds: BASE_SEED, BASE_SEED+1, …

ALPHAS        = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
LAYER_SIZES   = [4, 8, 12, 16, 20, 24, 28, 32, 48]

MAX_EPOCHS    = 100   # màxim d'epochs per run
PATIENCE      = 10    # early stopping: atura si val_loss no millora en N epochs
BATCH_SIZE    = 32

print(f"Quantitat de runs: {len(ALPHAS) * len(LAYER_SIZES) * N_RUNS}")
FAIRNESS_LOSS = "tpr_parity"  # tpr_parity | demographic_parity | disentanglement

Quantitat de runs: 270


In [4]:
# ── Gestió de l'experiment ──────────────────────────────────────────────────
RESULTS_BASE = REPO_DIR / "results" / "sweep_bias_accuracy"
RESULTS_BASE.mkdir(parents=True, exist_ok=True)

if experiment == "":
    EXP_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
    EXP_DIR = RESULTS_BASE / EXP_NAME
    EXP_DIR.mkdir(parents=True, exist_ok=True)

    config = {
        "N_RUNS": N_RUNS,
        "BASE_SEED": BASE_SEED,
        "ALPHAS": ALPHAS,
        "LAYER_SIZES": LAYER_SIZES,
        "MAX_EPOCHS": MAX_EPOCHS,
        "PATIENCE": PATIENCE,
        "BATCH_SIZE": BATCH_SIZE,
        "FAIRNESS_LOSS": FAIRNESS_LOSS,
    }
    with open(EXP_DIR / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"Nou experiment: {EXP_NAME}")

else:
    EXP_DIR = RESULTS_BASE / experiment
    if not EXP_DIR.exists():
        available = [d.name for d in RESULTS_BASE.iterdir() if d.is_dir()]
        raise FileNotFoundError(
            f"No s'ha trobat l'experiment '{experiment}' a {RESULTS_BASE}\n"
            f"Experiments disponibles: {available}"
        )
    with open(EXP_DIR / "config.json") as f:
        config = json.load(f)
    N_RUNS      = config["N_RUNS"]
    BASE_SEED   = config["BASE_SEED"]
    ALPHAS      = config["ALPHAS"]
    LAYER_SIZES = config["LAYER_SIZES"]
    MAX_EPOCHS  = config["MAX_EPOCHS"]
    PATIENCE    = config["PATIENCE"]
    BATCH_SIZE    = config["BATCH_SIZE"]
    FAIRNESS_LOSS = config.get("FAIRNESS_LOSS", "tpr_parity")
    print(f"Reprenent experiment: {experiment}")
    print(f"Config carregada: {config}")

RESULTS_FILE = str(EXP_DIR / "sweep_results_v2.json")
print(f"\nDirectori : {EXP_DIR}")
print(f"Resultats : {RESULTS_FILE}")

Reprenent experiment: 20260426_025714
Config carregada: {'N_RUNS': 5, 'BASE_SEED': 42, 'ALPHAS': [0.2, 0.4, 0.6, 0.8, 1.0, 1.2], 'LAYER_SIZES': [4, 8, 12, 16, 20, 24, 28, 32, 48], 'MAX_EPOCHS': 100, 'PATIENCE': 10, 'BATCH_SIZE': 32, 'FAIRNESS_LOSS': 'tpr_parity'}

Directori : /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260426_025714
Resultats : /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260426_025714/sweep_results_v2.json


In [5]:
# ── Sweep principal ─────────────────────────────────────────────────────────
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    print(f"Carregats {len(all_results)} runs previs de '{RESULTS_FILE}'")
else:
    all_results = []
    print("Fitxer de resultats nou.")

done = {(r['alpha'], r['ls'], r['run_id']) for r in all_results}

total_configs = len(ALPHAS) * len(LAYER_SIZES) * N_RUNS
remaining     = total_configs - len(done)
print(f"Total configuracions: {total_configs}  |  Pendents: {remaining}")

RETRY_WAIT_SECONDS = 15 * 60  # 15 minuts

for alpha in ALPHAS:
    for ls in LAYER_SIZES:
        for run_id in range(N_RUNS):
            if (alpha, ls, run_id) in done:
                print(f"[SKIP] alpha={alpha}, ls={ls}, run={run_id}")
                continue

            seed = BASE_SEED + run_id
            attempt = 0

            while True:
                attempt += 1
                print(f"\n{'='*60}")
                print(f"alpha={alpha}, ls={ls}, run={run_id}/{N_RUNS-1}, seed={seed}" +
                      (f"  [intent {attempt}]" if attempt > 1 else ""))
                print('='*60)

                try:
                    ctx = setup_experiment(
                        alpha=alpha,
                        prev2last_layer_size=ls,
                        batch_size=BATCH_SIZE,
                        seed=seed,
                        early_stopping_patience=PATIENCE,
                        run_name=f"sweep_a{alpha}_ls{ls}_r{run_id}",
                        results_base_dir=EXP_DIR / "results_NIH",
                        fairness_loss_fn=get_fairness_loss(FAIRNESS_LOSS),
                    )
                    set_all_seeds(seed)

                    result = fine_tune_and_evaluate(ctx, epochs=MAX_EPOCHS, alpha=alpha)

                    cm_all = result['All']['confusion_matrix']

                    cm_m = result['Male']['confusion_matrix']
                    tpr_male = cm_m[1][1] / (cm_m[1][1] + cm_m[1][0]) if (cm_m[1][1] + cm_m[1][0]) > 0 else float('nan')

                    cm_f = result['Female']['confusion_matrix']
                    tpr_female = cm_f[1][1] / (cm_f[1][1] + cm_f[1][0]) if (cm_f[1][1] + cm_f[1][0]) > 0 else float('nan')

                    tpr_gap = abs(tpr_male - tpr_female)

                    gender_pred = result.get('gender_pred', {})
                    last_layer_key = sorted(
                        [k for k in gender_pred.keys() if k.startswith('layer_')],
                        key=lambda x: int(x.split('_')[1])
                    )[-1] if gender_pred else None
                    gender_auc_last = gender_pred[last_layer_key]['auc'] if last_layer_key else float('nan')

                    run_result = {
                        'alpha'               : alpha,
                        'ls'                  : ls,
                        'run_id'              : run_id,
                        'seed'                : seed,
                        'epochs_trained'      : ctx['total_epochs'],
                        'accuracy'            : result['All']['accuracy'],
                        'auc_all'             : result['All']['auc'],
                        'tpr_male'            : tpr_male,
                        'tpr_female'          : tpr_female,
                        'tpr_gap'             : tpr_gap,
                        'gender_auc_last_layer': gender_auc_last,
                        'results_dir'         : ctx['results_dir'],
                    }
                    all_results.append(run_result)
                    done.add((alpha, ls, run_id))

                    with open(RESULTS_FILE, 'w') as f:
                        json.dump(all_results, f, indent=2)
                    print(f"  Run guardat ({ctx['total_epochs']} epochs) a '{RESULTS_FILE}'")
                    break  # èxit: passa al següent run

                except Exception as e:
                    if 'out of memory' in str(e).lower():
                        print(f"  [OOM] CUDA sense memòria (intent {attempt}): {e}")
                        gc.collect()
                        torch.cuda.empty_cache()
                        print(f"  Esperant {RETRY_WAIT_SECONDS // 60} minuts abans de reintentar...")
                        time.sleep(RETRY_WAIT_SECONDS)
                    else:
                        print(f"  ERROR inesperat: {e}")
                        import traceback; traceback.print_exc()
                        break  # errors no-OOM no es reintenten

                finally:
                    if 'ctx' in dir():
                        del ctx
                    gc.collect()
                    torch.cuda.empty_cache()

print("\nSweep completat!")


Carregats 92 runs previs de '/home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260426_025714/sweep_results_v2.json'
Total configuracions: 270  |  Pendents: 178
[SKIP] alpha=0.2, ls=4, run=0
[SKIP] alpha=0.2, ls=4, run=1
[SKIP] alpha=0.2, ls=4, run=2
[SKIP] alpha=0.2, ls=4, run=3
[SKIP] alpha=0.2, ls=4, run=4
[SKIP] alpha=0.2, ls=8, run=0
[SKIP] alpha=0.2, ls=8, run=1
[SKIP] alpha=0.2, ls=8, run=2
[SKIP] alpha=0.2, ls=8, run=3
[SKIP] alpha=0.2, ls=8, run=4
[SKIP] alpha=0.2, ls=12, run=0
[SKIP] alpha=0.2, ls=12, run=1
[SKIP] alpha=0.2, ls=12, run=2
[SKIP] alpha=0.2, ls=12, run=3
[SKIP] alpha=0.2, ls=12, run=4
[SKIP] alpha=0.2, ls=16, run=0
[SKIP] alpha=0.2, ls=16, run=1
[SKIP] alpha=0.2, ls=16, run=2
[SKIP] alpha=0.2, ls=16, run=3
[SKIP] alpha=0.2, ls=16, run=4
[SKIP] alpha=0.2, ls=20, run=0
[SKIP] alpha=0.2, ls=20, run=1
[SKIP] alpha=0.2, ls=20, run=2
[SKIP] alpha=0.2, ls=20, run=3
[SKIP] alpha=0.2, ls=20, run=4
[SKIP] alpha=0.2, ls=24, run=0
[SKIP] a

Device: cuda
Config: alpha=0.6, layer_size=4, batch_size=32, lr=0.0001


Model: FeatureExtractor (no dropout)
Frozen params: 11,176,512  |  Trainable params: 2,062
Data: 1829 train / 458 test samples
Results dir: /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260426_025714/results_NIH/sweep_a0.6_ls4_r2_20260426_141440


  Loss: 0.7131 + 0.6*fairness penalty: 0.0009 # Loss gradient norms: 1.5749 # CE-Loss grad. norms: 1.5742


  Epoch 1, Train Loss: 0.7136, Val Loss: 0.6665


...

  Layer 1 - Gender prediction AUC: 1.000


  Layer 2 - Gender prediction AUC: 1.000


  Layer 3 - Gender prediction AUC: 1.000


  Layer 4 - Gender prediction AUC: 1.000
  Layer 5 - Gender prediction AUC: 1.000
  Layer 6 - Gender prediction AUC: 0.849
  Run guardat (39 epochs) a '/home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260426_025714/sweep_results_v2.json'



Sweep completat!
